# PolyMelt · A polydisperse polyethylene melt, from G²RINS to MARTINI 3

**CHEM 73000 · Polymer Chemistry (Part 2) · CUNY · Lecture 5 hands-on · Prof. Sharon Loverde**

<a href="https://colab.research.google.com/github/sloverde/PolyMelt/blob/main/Polyethylene_MARTINI3_G2RINS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In Lecture 5 we saw that the right simulation method depends on the length and time scales you care about.
In **Dr. Gervasio Zaldivar's guest lecture** you used **G²RINS** to write a whole *mixture* of polymer chains as one line of text,
and his last question was: *how could these ensembles be used to set up a molecular-dynamics simulation?* Today you do exactly that:

**G²RINS string → a polydisperse sample of chains → a MARTINI 3 coarse-grained model → a GROMACS simulation → a movie and diffusion coefficients.**

| Part | What you do | About |
|---|---|---|
| 1 | Install the tools | 3 min |
| A | From atoms to beads: the all-atom C<sub>20</sub> melt and its MARTINI 3 version | 5 min |
| B | Build a polydisperse polyethylene sample with G²RINS | 5 min |
| C | Write the MARTINI 3 model and pack the chains in a box | 3 min |
| D | Run GROMACS: minimize → compress → equilibrate → production | 10 min |
| E | Visualize the melt in 3D | 5 min |
| F | Measure chain size and diffusion: R<sub>g</sub>(N), MSD(t), D(N) | 10 min |

Short on time? Every analysis also works with a **precomputed 40 ns run** of exactly the same system (see Part D).

## 0 · How to use this notebook

* Gray boxes are **code cells**. Click one and press **Shift + Enter** (or the ▶ button) to run it. Run them **in order**.
* If Colab warns *"This notebook was not authored by Google"*, click **Run anyway**.
* Cells marked **✏️ Your turn** have a few numbers or words you can change.
* Nothing can break. If something goes wrong: **Runtime → Restart session**, then run the cells again from the top.
* To keep your work: **File → Save a copy in Drive**.

## 1 · Install GROMACS, G²RINS and the analysis tools (run once, about 3 minutes)

This installs the **GROMACS** molecular-dynamics engine, Dr. Zaldivar's **`g2rins`** package (v1.0.0, the same one you used in the guest lecture),
**MDAnalysis** for analysis and **py3Dmol** for 3D graphics, then downloads this repository. A lot of text may scroll by; that is normal.

In [ ]:
# Install GROMACS, G2RINS (v1.0.0), MDAnalysis and py3Dmol, then download this repository
import os, sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    !apt-get -qq update > /dev/null && apt-get -qq install -y gromacs > /dev/null
    !pip -q install "g2rins @ git+https://github.com/depablolab/g2rins.git@v1.0.0" MDAnalysis py3Dmol
    if not os.path.isdir("/content/PolyMelt"):
        !git clone -q https://github.com/sloverde/PolyMelt.git /content/PolyMelt
    %cd /content/PolyMelt
sys.path.insert(0, os.path.abspath("martini3"))
!gmx --version 2>/dev/null | grep "GROMACS version"
print("Setup finished!")

In [ ]:
# @title Load the helper tools (just run it) { display-mode: "form" }
import json, shutil, subprocess, time
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Image, display
import pe_tools as pt

plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False, "font.size": 11})

def gmx(args, log):
    """Run one GROMACS command; show the end of the log only if something goes wrong."""
    r = subprocess.run("gmx " + args, shell=True, capture_output=True, text=True)
    open(log, "w").write(r.stdout + r.stderr)
    if r.returncode != 0:
        print((r.stdout + r.stderr)[-2500:])
        raise RuntimeError(f"gmx {args.split()[0]} failed; see {log}")
print("Helpers loaded.")

## Part A · From atoms to beads

The original system in this repository (built with CHARMM-GUI, CHARMM general force field; notebook `Polyethylene2_2023.ipynb`)
is a melt of 1,286 eicosane chains, C<sub>20</sub>H<sub>42</sub>, with **62 atoms per chain**.

**MARTINI 3** lumps about four heavy atoms and their hydrogens into one bead. For polyethylene, every **4 CH<sub>2</sub> groups become one "C1" bead**,
the same bead MARTINI 3 uses for octane (2 beads), dodecane (3) and hexadecane (4). So one C<sub>20</sub> chain (62 atoms) becomes **5 beads**.

Below: one all-atom C<sub>20</sub> chain from the repository (sticks; gray = carbon, white = hydrogen) with its five MARTINI 3 beads drawn on top.
Drag to rotate, scroll to zoom.

In [ ]:
pt.show_mapping("step3_input.gro", chain=0).show()

> **Question A.1** · A box of 1,000 C<sub>20</sub> chains has how many all-atom particles? How many MARTINI 3 beads?
>
> **Question A.2** · Why can a coarse-grained model use a 20 fs time step when an all-atom model needs 2 fs? (Think about the fastest motion left in each model; compare In-Class Example 5.5.)
>
> *Your answers:* (double-click to type)

## Part B · A polydisperse sample with G²RINS

Real polyethylene is never one chain length. Here is polyethylene in G²RINS:

```
{[] [<]CC[>]; [>][H]; [<][H] []}|schulz_zimm(2400, 1200)|
```

| Piece | Meaning |
|---|---|
| `[<]CC[>]` | the repeat unit, –CH<sub>2</sub>–CH<sub>2</sub>– (28 g/mol) |
| `[>][H]` and `[<][H]` | the two chain ends are capped with H, so every chain is a linear alkane |
| `schulz_zimm(2400, 1200)` | molecular-weight distribution with **M<sub>w</sub> = 2400** and **M<sub>n</sub> = 1200 g/mol**, so **Đ = 2**, typical of a broad industrial sample |

The `g2rins` package turns this one line into as many chains as you want, exactly as in the guest-lecture notebook.

In [ ]:
# ✏️ Your turn (optional): change the distribution or the number of chains, then rerun Parts B–F.
pe_g2rins = "{[] [<]CC[>]; [>][H]; [<][H] []}|schulz_zimm(2400, 1200)|"  # @param {type:"string"}
n_chains = 200   # @param {type:"integer"}
seed = 2026      # @param {type:"integer"}

sample = pt.sample_chains(pe_g2rins, n_chains=n_chains, seed=seed)
beads = pt.martini3_beads(sample["n_carbons"])
print(f"{len(beads)} chains   Mn = {sample['Mn']:.0f} g/mol   Mw = {sample['Mw']:.0f} g/mol   Đ = {sample['D']:.2f}")
print(f"chain lengths: {beads.min()} to {beads.max()} MARTINI 3 beads ({sample['n_carbons'].min()} to {sample['n_carbons'].max()} carbons)")
print(f"total beads in the box: {beads.sum()}")

fig, ax = plt.subplots(1, 2, figsize=(10, 3.6))
ax[0].hist(sample["mw"], bins=30, color="#1B7F79", alpha=0.85)
ax[0].axvline(sample["Mn"], color="#C97F00", lw=2, label=f"Mn = {sample['Mn']:.0f}")
ax[0].axvline(sample["Mw"], color="#17324D", lw=2, ls="--", label=f"Mw = {sample['Mw']:.0f}")
ax[0].set_xlabel("chain molecular weight (g/mol)"); ax[0].set_ylabel("number of chains"); ax[0].legend()
ax[1].hist(beads, bins=np.logspace(np.log10(2), np.log10(beads.max() + 1), 20), color="#7c3aed", alpha=0.8)
ax[1].set_xscale("log"); ax[1].set_xlabel("chain length N (MARTINI 3 beads)"); ax[1].set_ylabel("number of chains")
fig.suptitle("Your G²RINS sample"); fig.tight_layout(); plt.show()

Chains shorter than about C<sub>10</sub> are given the smallest MARTINI 3 alkane, 2 beads (like octane).

Try `poisson(1200)` (ideal living polymerization, Đ ≈ 1) or `schulz_zimm(4800, 1200)` (Đ = 4) in the cell above.
If you change the sample, your own simulation in Part D will use it; the precomputed run always uses the default string and seed.

> **Question B.1** · Why does a sample with Đ = 2 contain so many very short chains? Which chains carry most of the *mass*?
>
> *Your answer:*

## Part C · Write the MARTINI 3 model and pack the box

`pe_tools` now writes everything GROMACS needs into the folder `my_run/`:

* **`PE_chains.itp`** · one molecule type per chain length (`PE5` is a 5-bead chain, i.e. C<sub>20</sub>). Bonded parameters are those of the official MARTINI 3 alkanes:
  bonds 0.47 nm (0.475 nm at chain ends) with k = 3800 kJ mol<sup>−1</sup> nm<sup>−2</sup>, and angles of 180° with k = 35 kJ mol<sup>−1</sup>.
* **`martini_v3.0.0_C1.itp`** · the non-bonded part of the force field. Polyethylene needs only **one** MARTINI 3 bead type, C1, so this
  file is just one Lennard-Jones pair: σ = 0.47 nm, ε = 3.39 kJ/mol (copied unchanged from the official MARTINI 3 file).
* **`topol.top`** · the system: the force field plus how many chains of each length.
* **`start.gro`** · starting coordinates. In a melt every chain is an ideal random walk (Flory), so each chain is grown as a random walk
  with the bending stiffness of the MARTINI 3 angle potential, directly at melt density. Beads that start too close are pushed apart by the energy minimization in Part D.

In [ ]:
run = "my_run"
os.makedirs(run, exist_ok=True)
shutil.copy("martini3/martini_v3.0.0_C1.itp", run)
order = pt.write_topology(beads, run)
box = pt.build_melt(order, f"{run}/start.gro", seed=seed)
np.savetxt(f"{run}/chain_lengths.txt", order, fmt="%d", header="beads per chain, in start.gro order")
print(f"Packed {len(order)} chains ({sum(order)} beads) in a {box:.2f} nm box.\n")
print(open(f"{run}/topol.top").read()[:400], "...\n")
itp = open(f"{run}/PE_chains.itp").read()
k = itp.find("PE5 ")
print(itp[itp.rfind("[ moleculetype ]", 0, k):itp.find("[ moleculetype ]", k + 1)] if k > 0 else itp[:900])

In [ ]:
# The starting box: random-walk chains at melt density, colored by length (purple = short, yellow = long)
pt.show_melt(pt.load_run(f"{run}/start.gro")).show()

## Part D · Run GROMACS

Four short runs, each starting from the end of the previous one (all at **450 K**, above the melting point of polyethylene, and 1 bar):

| Step | File | What it does |
|---|---|---|
| 1 | `em.mdp` | energy minimization: removes overlapping beads |
| 2 | `npt_relax.mdp` | 0.5 ns at constant pressure with a short 10 fs step: beads settle and the density adjusts |
| 3 | `npt_equil.mdp` | 2 ns with the full **20 fs** MARTINI time step |
| 4 | `md.mdp` | production run used for the analysis, positions saved every 20 ps |

On a 2-core machine like free Colab, GROMACS runs the default sample (about 4,600 beads) at roughly **2,000–2,500 ns per day**, so a 10 ns production takes 5–10 minutes.

In [ ]:
# ✏️ Your turn: length of your production run (ns). 10 ns takes 5–10 min; 2 ns takes 1–2 min.
production_ns = 10  # @param {type:"number"}

mdp = open("martini3/mdp/md.mdp").read()
nsteps = int(round(production_ns * 1000 / 0.020))
mdp = "\n".join((f"nsteps                   = {nsteps}" if l.startswith("nsteps") else l) for l in mdp.splitlines())
open(f"{run}/md.mdp", "w").write(mdp + "\n")

t0 = time.time()
os.chdir(run)
try:
    steps = [("minimization", "../martini3/mdp/em.mdp", "start.gro", "em"),
             ("relaxation", "../martini3/mdp/npt_relax.mdp", "em.gro", "relax"),
             ("equilibration", "../martini3/mdp/npt_equil.mdp", "relax.gro", "equil"),
             (f"production ({production_ns} ns)", "md.mdp", "equil.gro", "md")]
    for name, mdp, gro, out in steps:
        # each run starts from the last structure of the previous one (with its velocities);
        # beads are first put back inside the box, which older GROMACS versions need (issue 5002)
        pt.wrap_gro(gro, f"{out}_in.gro")
        gmx(f"grompp -f {mdp} -c {out}_in.gro -p topol.top -o {out}.tpr", "grompp.log")
        pt.check_tpr(f"{out}.tpr")
        gmx(f"mdrun -deffnm {out}", "mdrun.log")
        print(f"✓ {name:22s} done after {time.time() - t0:5.0f} s")
    # GROMACS puts every bead back in the box, so chains get cut at the box edge and jump across it.
    # -pbc nojump undoes the jumps. Its reference must have whole chains: equil.gro (mdrun writes its last frame whole).
    r = subprocess.run("echo 0 | gmx trjconv -s equil.gro -f md.xtc -o md_nojump.xtc -pbc nojump",
                       shell=True, capture_output=True, text=True)
    open("trjconv.log", "w").write(r.stdout + r.stderr)
    if not os.path.exists("md_nojump.xtc"):
        raise RuntimeError("trjconv failed; see my_run/trjconv.log")
    print("✓ trajectory unwrapped (md_nojump.xtc)")
finally:
    os.chdir("..")

**Check: did the box reach melt density?** MARTINI gives every C1 bead a mass of 72 g/mol, but a real C<sub>4</sub>H<sub>8</sub> group is 56.1 g/mol,
so we rescale the density GROMACS reports. For comparison, long-chain polyethylene at 450 K has a density of about 0.77 g/cm³;
a sample with many short chains is a little lighter, because every chain end adds free volume.

In [ ]:
def density_xvg(edr):
    out = edr.replace(".edr", "_density.xvg")
    subprocess.run(f"echo Density | gmx energy -f {edr} -o {out}", shell=True, capture_output=True, text=True)
    d = np.loadtxt(out, comments=("#", "@"))
    return d[:, 0] / 1000.0, d[:, 1] * 56.11 / 72.0 / 1000.0     # ns, g/cm3 with real masses

fig, ax = plt.subplots(figsize=(7.5, 3.4))
t_off = 0.0
for stage, col in [("relax", "#C97F00"), ("equil", "#1B7F79"), ("md", "#7c3aed")]:
    edr = f"{run}/{stage}.edr"
    if os.path.exists(edr):
        tt, rho = density_xvg(edr)
        ax.plot(tt - tt[0] + t_off, rho, color=col, lw=1.4, label=stage)
        t_off += tt[-1] - tt[0]
ax.axhline(0.77, color="gray", ls=":", lw=1)
ax.text(0.02, 0.775, "long-chain PE melt, 450 K", color="gray", fontsize=9, transform=ax.get_yaxis_transform())
ax.set_xlabel("time (ns)"); ax.set_ylabel("density (g/cm³)"); ax.legend(loc="lower right")
ax.set_title("Density during relaxation, equilibration and production"); plt.show()
print(f"average density in the production run: {rho[len(rho) // 2:].mean():.3f} g/cm³")

> **Question D.1** · Is your melt lighter or denser than long-chain polyethylene (0.77 g/cm³)? Give two possible reasons.
> (Hints: how many chain ends does your sample have? At what temperature were the MARTINI 3 parameters tuned?)
>
> *Your answer:*

In [ ]:
# ✏️ Choose which trajectory to analyze
use_precomputed = False  # @param {type:"boolean"}

if use_precomputed or not os.path.exists(f"{run}/md_nojump.xtc"):
    data = "martini3/precomputed"
    print("Using the precomputed 40 ns run:", json.load(open(f"{data}/sample.json"))["g2rins"])
else:
    data = run
    print("Using your own run in", run)
u = pt.load_run(f"{data}/start.gro", f"{data}/md_nojump.xtc")
sizes, _ = pt.chain_info(u)
print(f"{len(sizes)} chains, {u.atoms.n_atoms} beads, {len(u.trajectory)} frames, {u.trajectory[-1].time / 1000:.1f} ns")

## Part E · Look at the melt

Every chain is colored by its length: **purple = short, orange = medium, yellow = long**. The light-blue frame is the periodic box.
Drag to rotate, scroll to zoom.

In [ ]:
view = pt.show_melt(u, frame=-1, spin=True)
view.show()

Hard to follow one chain in a crowd! Below, everything is ghosted except the **3 shortest**, the **median** and the **3 longest** chains.

In [ ]:
rank = np.argsort(sizes)
picked = list(rank[:3]) + [rank[len(rank) // 2]] + list(rank[-3:])
print("highlighted chain lengths (beads):", [int(sizes[c]) for c in picked])
pt.show_melt(u, frame=-1, highlight=picked).show()

### A movie: who moves farther?

Two short chains and two long chains over the whole run. Positions are **unwrapped**, so the chains can leave the box: you see their true paths.

In [ ]:
movie_chains = list(rank[:2]) + list(rank[-2:])
pt.animate_chains(u, movie_chains, stride=max(1, len(u.trajectory) // 60)).show()

In [ ]:
t, com = pt.chain_com(u)
ax = pt.plot_com_paths(t, com, sizes, list(rank[:3]) + list(rank[-3:]))
plt.show()

In [ ]:
# A poster-style render you can download for slides or a report (saved as melt.png)
display(Image(pt.render_melt_png(u, "melt.png")))

## Part F · Measure it

### F.1 · How big is a chain? R<sub>g</sub> versus N

First a check every simulator should make: **is the melt equilibrated?** If it is, every chain keeps the same average size during the run,
so the curves below are flat. (Long chains relax slowly, with a relaxation time that grows as N², so they are the first to drift.)

In a melt each chain is surrounded by chains just like itself, so excluded volume is screened and long chains are **ideal random walks**
(Flory's theorem, Lecture 1): ⟨R<sub>g</sub><sup>2</sup>⟩ ∝ N. Over a few beads, though, a chain is fairly stiff, like a short rod, and for a rod
R<sub>g</sub><sup>2</sup> ∝ N². Look for the crossover from one to the other.

In [ ]:
t_rg, n_rg, rg2_t = pt.rg2_time(u, step=5)
rg2 = rg2_t.mean(axis=0)
fig, ax = plt.subplots(1, 2, figsize=(11.5, 4.2))
cm = plt.colormaps["plasma"]
glist = [(lo, hi) for lo, hi in [(5, 9), (9, 17), (17, 33), (33, 65), (65, 10**6)] if np.sum((n_rg >= lo) & (n_rg < hi)) >= 3]
for k, (lo, hi) in enumerate(glist):
    sel = (n_rg >= lo) & (n_rg < hi)
    y = rg2_t[:, sel].mean(axis=1)
    w = max(1, len(y) // 20)                                   # running average over 5 % of the run
    ax[0].plot(t_rg[w - 1:], np.convolve(y, np.ones(w) / w, "valid") / y.mean(), color=cm(0.08 + 0.84 * k / max(1, len(glist) - 1)),
               lw=2, label=f"N = {lo}–{min(hi - 1, n_rg.max())}")
ax[0].axhline(1, color="gray", ls=":", lw=1); ax[0].set_ylim(0.6, 1.4)
ax[0].set_xlabel("time (ns)"); ax[0].set_ylabel(r"$\langle R_g^2 \rangle(t)$ / run average"); ax[0].legend(fontsize=8)
ax[0].set_title("Equilibrated? (flat = yes)")
ax[1].scatter(n_rg, rg2, c=np.log(n_rg), cmap="plasma", s=22, edgecolors="none")
small = n_rg <= 4
nn = np.array([2, 6]); ax[1].plot(nn, np.exp(np.mean(np.log(rg2[small] / n_rg[small] ** 2))) * nn ** 2, "k--", lw=1.2, label="rod: slope 2")
nn = np.array([20, n_rg.max()]); big = n_rg >= 20
ax[1].plot(nn, np.exp(np.mean(np.log(rg2[big] / n_rg[big]))) * nn, "k:", lw=1.5, label="ideal coil: slope 1")
ax[1].set_xscale("log"); ax[1].set_yscale("log"); ax[1].set_xlabel("N (beads)"); ax[1].set_ylabel(r"$\langle R_g^2 \rangle$ (nm$^2$)")
ax[1].legend(); ax[1].set_title("Chain size: from rod to coil"); fig.tight_layout(); plt.show()
for lo, hi in [(3, 8), (20, n_rg.max() + 1)]:
    s_ = (n_rg >= lo) & (n_rg < hi)
    print(f"local exponent for N = {lo}–{hi - 1}: ⟨Rg²⟩ ∝ N^{np.polyfit(np.log(n_rg[s_]), np.log(rg2[s_]), 1)[0]:.2f}")

### F.2 · How do chains move? The mean-squared displacement

The MSD of a chain's centre of mass, ⟨|**r**(t+τ) − **r**(t)|²⟩, is averaged over every time origin t (computed with an FFT) and over all chains of similar length.
On a log–log plot the **slope** tells you the regime: **1 = free diffusion** (Einstein: MSD = 6 D τ), less than 1 = subdiffusive.

For one bead in the *middle* of a long chain, the Rouse model (Lecture 1) predicts a characteristic **τ<sup>1/2</sup>** regime: the bead is tied to its neighbors and can only move as fast as the whole chain section around it relaxes.

In [ ]:
msd = pt.msd_fft(com)                       # (lag, chain) in nm²; lag time = t
half = t <= 0.5 * t[-1]                     # beyond half the run there are too few time origins
bins = [2, 3, 5, 9, 17, 33, 65, 129, 10**6]
groups = [(lo, hi) for lo, hi in zip(bins[:-1], bins[1:]) if np.sum((sizes >= lo) & (sizes < hi)) >= 3]
cmap = plt.colormaps["plasma"]
fig, ax = plt.subplots(1, 2, figsize=(11.5, 4.3))
for k, (lo, hi) in enumerate(groups):
    sel = (sizes >= lo) & (sizes < hi)
    m = msd[:, sel].mean(axis=1)
    col = cmap(0.08 + 0.84 * k / max(1, len(groups) - 1))
    lab = f"N = {lo}" if hi - lo == 1 else f"N = {lo}–{min(hi - 1, sizes.max())}"
    ax[0].loglog(t[1:][half[1:]], m[1:][half[1:]], color=col, lw=2, label=f"{lab} ({sel.sum()} chains)")
    ts, sl = pt.local_slope(t, m)
    ax[1].semilogx(ts, sl, color=col, lw=1.8)
# a slope-1 guide placed under the curves
tt = np.array([t[1], 0.5 * t[-1]])
m_long = msd[:, (sizes >= groups[-1][0]) & (sizes < groups[-1][1])].mean(axis=1)
ax[0].loglog(tt, 0.3 * m_long[len(t) // 4] * tt / t[len(t) // 4], "k--", lw=1, label="slope 1 (free diffusion)")
ax[0].set_xlabel("lag time τ (ns)"); ax[0].set_ylabel("centre-of-mass MSD (nm²)"); ax[0].legend(fontsize=8)
ax[0].set_title("Short chains move farther")
for y, txt in [(1, "free diffusion"), (0.5, "Rouse")]:
    ax[1].axhline(y, color="gray", ls=":", lw=1); ax[1].text(t[1], y + 0.03, txt, color="gray", fontsize=9)
ax[1].set_ylim(0, 1.5); ax[1].set_xlabel("lag time τ (ns)"); ax[1].set_ylabel("local slope  d ln MSD / d ln τ")
ax[1].set_title("Where does each group reach slope 1?"); fig.tight_layout(); plt.show()

In [ ]:
# Middle bead (g1) versus centre of mass (g3) for the longest chains
tb, mid = pt.bead_positions(u, "middle")
g1 = pt.msd_fft(mid)
cut = int(np.percentile(sizes, 85))
longest = sizes >= cut
g1m, g3m = g1[:, longest].mean(axis=1), msd[:, longest].mean(axis=1)
fig, ax = plt.subplots(figsize=(6, 4.3))
ax.loglog(tb[1:][half[1:]], g1m[1:][half[1:]], color="#C97F00", lw=2.2, label="middle bead, $g_1$")
ax.loglog(t[1:][half[1:]], g3m[1:][half[1:]], color="#7c3aed", lw=2.2, label="centre of mass, $g_3$")
ax.loglog(tb[1:6], 1.6 * g1m[1] * (tb[1:6] / tb[1]) ** 0.5, "k:", lw=1.5, label=r"$\tau^{1/2}$ (Rouse)")
ax.loglog(tb[len(tb) // 8:len(tb) // 2], 0.6 * g3m[len(tb) // 8] * (tb[len(tb) // 8:len(tb) // 2] / tb[len(tb) // 8]),
          "k--", lw=1, label=r"$\tau^{1}$ (diffusion)")
ax.set_xlabel("lag time τ (ns)"); ax.set_ylabel("MSD (nm²)")
ax.set_title(f"Longest chains (N ≥ {cut} beads, {longest.sum()} chains)"); ax.legend(); plt.show()

> The middle bead first moves *faster* than the whole chain, but with a slope of about ½: it is held back by its neighbors.
> Once the lag time is longer than the chain's slowest (Rouse) relaxation time, the bead can only move by carrying the whole chain with it, and **g₁ joins g₃**.

### F.3 · Diffusion coefficient versus chain length

We fit MSD = 6 D τ over the diffusive window (10–40 % of the run, where every point averages many time origins).
Error bars are bootstrap 68 % intervals over the chains in each group. **Open symbols** mark groups whose chains have not yet moved farther than
their own size (MSD < 2⟨R<sub>g</sub><sup>2</sup>⟩); their D is only an upper estimate, so run longer (or use the precomputed 40 ns run) to pin it down.

The **Rouse model** (unentangled chains, friction ∝ N) predicts **D ∝ N<sup>−1</sup>**.

In [ ]:
res = pt.diffusion_by_length(t, msd, sizes, rg2=rg2)
print(f"{'N range':>9s} {'<N>':>6s} {'chains':>6s} {'D (nm²/ns)':>11s} {'D (cm²/s)':>10s}  moved > own size?")
for r in res:
    print(f"{r['N_lo']:>3d}–{r['N_hi']:<5d} {r['N']:6.1f} {r['chains']:6d} {r['D']:11.4f} {r['D'] * 1e-5:10.2e}  {'yes' if r['diffusive'] else 'not yet'}")

N = np.array([r["N"] for r in res]); D = np.array([r["D"] for r in res])
err = np.array([[r["D"] - r["D_lo"] for r in res], [r["D_hi"] - r["D"] for r in res]])
ok = np.array([r["diffusive"] for r in res])
fit = ok & (N >= 5)
fig, ax = plt.subplots(figsize=(6.2, 4.5))
for i in range(len(N)):
    col = plt.colormaps["plasma"](0.08 + 0.84 * i / max(1, len(N) - 1))
    ax.errorbar(N[i], D[i], yerr=err[:, i:i + 1], fmt="o", ms=8, capsize=3, color=col, mec=col,
                mfc=col if ok[i] else "white")
if fit.sum() >= 2:
    p, c = np.polyfit(np.log(N[fit]), np.log(D[fit]), 1)
    nn = np.array([N[fit].min(), N[fit].max()])
    ax.plot(nn, np.exp(c) * nn ** p, color="#17324D", lw=2, label=rf"fit (N ≥ 5): D $\propto$ N$^{{{p:.2f}}}$")
    print(f"\nD ∝ N^{p:.2f} for N ≥ 5   (Rouse model: N^-1)")
i5 = np.argmin(np.abs(N - 6))
nr = np.array([N[i5], N.max()])
ax.plot(nr, 1.3 * D[i5] * (nr / N[i5]) ** -1.0, "k--", lw=1, label=r"Rouse: D $\propto$ N$^{-1}$")
ax.set_xscale("log"); ax.set_yscale("log"); ax.set_xlabel("chain length N (beads)"); ax.set_ylabel("D (nm²/ns)")
ax.set_xticks([t_ for t_ in (2, 5, 10, 20, 50, 100) if t_ <= 1.5 * N.max()])
ax.xaxis.set_major_formatter(plt.matplotlib.ticker.ScalarFormatter()); ax.xaxis.set_minor_formatter(plt.matplotlib.ticker.NullFormatter())
ax.set_title("Diffusion in the polydisperse melt\n(open symbols: chains have not yet moved farther than their own size)", fontsize=10)
ax.legend(); plt.show()

> **Question F.1** · What exponent did you find for D(N)? Is it close to the Rouse value −1? What would you expect for chains long enough to be **entangled** (reptation)?
>
> **Question F.2** · Why are the error bars largest for the longest chains? (Count the chains in each group, and look at how far they moved.)
>
> **Question F.3** · Convert one D to cm²/s (1 nm²/ns = 10⁻⁵ cm²/s). MARTINI dynamics run faster than real time because the smooth bead–bead potentials remove friction. How could you calibrate the time scale against experiment or an all-atom simulation?
>
> **Question F.4** · An NMR experiment measures one *average* D for the whole sample. Which chains dominate that average in a sample with Đ = 2?
>
> *Your answers:*

## Optional · The all-atom model

The original all-atom workflow (CHARMM36, C<sub>20</sub>H<sub>42</sub>, 79,732 atoms) is still in this repository:
<a href="https://colab.research.google.com/github/sloverde/PolyMelt/blob/main/Polyethylene2_2023.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>
Compare its time step (2 fs) and its 79,732 atoms with the 20 fs step and the roughly 4,600 beads you ran today.

## Wrap-up

1. What did G²RINS give you that a single SMILES string could not?
2. Which result depends on polydispersity: R<sub>g</sub>(N), D(N), or the sample-average D?
3. What would you change to study an **entangled** polyethylene melt?

### Credits

* **G²RINS**: G. Zaldivar, Y. Tian, G. Sun, C. Kappatou, P. Eiden, N. B. Wulkow, V. Settels, J. J. de Pablo, ChemRxiv (2026), [doi:10.26434/chemrxiv.15007504/v1](https://doi.org/10.26434/chemrxiv.15007504/v1); code: <https://github.com/depablolab/g2rins> (GPL-3.0). Guest lecture and hands-on: <https://github.com/sloverde/CHEM73000-G2RINS-HandsOn>.
* **MARTINI 3**: P. C. T. Souza *et al.*, *Nat. Methods* **18**, 382 (2021); force-field files from <https://github.com/marrink-lab/martini-forcefields> (Apache-2.0).
* **GROMACS** (<https://www.gromacs.org>), **MDAnalysis** (<https://www.mdanalysis.org>), **py3Dmol** (<https://3dmol.org>).
* All-atom system: CHARMM-GUI (<https://www.charmm-gui.org>).